In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms
from datasets import load_dataset
import matplotlib.pyplot as plt

# Usar GPU si esta disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

/home/uno21/.virtualenvs/traffic-sign-cnn/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando: cuda


In [3]:
ds = load_dataset("tanganke/gtsrb")

In [4]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3337, 0.3064, 0.3171],
                         std=[0.2672, 0.2564, 0.2629])
])

transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3337, 0.3064, 0.3171],
                         std=[0.2672, 0.2564, 0.2629])
])

In [5]:
# Dataset wrapper
class GTSRBDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.data = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]['image'].convert('RGB')
        label = self.data[idx]['label']
        if self.transform:

            image = self.transform(image)
        return image, label

In [6]:
# Splits
full_train = GTSRBDataset(ds['train'], transform=transform_train)
test_dataset = GTSRBDataset(ds['test'], transform=transform)

train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
train_dataset, val_dataset = random_split(full_train, [train_size, val_size])

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 21312
Val: 5328
Test: 12630


In [7]:
# DataLoaders
# V.2 - Usar num_workers y pin_memory para mejorar la velocidad de carga en GPU
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Batches en train: {len(train_loader)}")
print(f"Batches en val: {len(val_loader)}")
print(f"Batches en test: {len(test_loader)}")

Batches en train: 333
Batches en val: 84
Batches en test: 198


In [8]:
class CNNBaseline(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 32x32 -> 16x16
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units*2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 16x16 -> 8x8
        )

        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units*2, out_channels=hidden_units*4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 8x8 -> 4x4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(in_features=hidden_units*4 * 4 * 4, out_features=256),
            nn.ReLU(),
            nn.Linear(256, output_shape)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.classifier(x)
        return x

In [9]:
torch.manual_seed(42)
model = CNNBaseline(input_shape=3, hidden_units=32, output_shape=43).to(device)

In [10]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")

Total parámetros: 628,843
Parámetros entrenables: 628,843


In [11]:
loss_fn = nn.CrossEntropyLoss()  # Dice q tan mal el modelo esta
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # es cuanto se va a actualizar el modelo o corregir

In [12]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    return (correct / len(y_pred)) * 100

In [13]:
from tqdm.auto import tqdm

def print_train_time(start, end, device=None):
    total_time = end - start
    print(f"Tiempo de entrenamiento: {total_time:.3f} segundos en {device}")
    return total_time

In [14]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
    """ Performs a training with model tryinh to learn on dta_loader."""
    train_loss, train_acc = 0,0

    # put model into training mode
    model.train()

    # add a loop to loop through the training batches
    for batch, (X, y) in enumerate(data_loader): # X, y can also be named "images, labels"
        # Send data to GPU
        X, y = X.to(device), y.to(device)

        # 1. forward pass (outputs the raw logits from the model)
        y_pred = model(X)

        # 2. Calculate loss and accuracy (per batch)
        loss = loss_fn(y_pred, y)
        train_loss += loss # accumulate train loss
        train_acc += accuracy_fn(y_true=y,
                                 y_pred=y_pred.argmax(dim=1)) # go from logits -> prediction labels

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()
    # Divide total train loss by lenght of train dataloader
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    print(f"Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%")

In [15]:
def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device =device):
    """ Performs a testing loop step on model over data_loader"""
    test_loss, test_acc = 0,0

    #put model on eval mode
    model.eval()

    # Turn on inference  mode context manager
    with torch.inference_mode():
        for X, y in data_loader:
            # Send the data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            test_pred = model(X)

            #Calculate the loss /acc
            test_loss += loss_fn(test_pred,y)
            test_acc += accuracy_fn(y_true=y,
                                    y_pred=test_pred.argmax(dim=1)) # go from logits -> prediction labels

        # Adjust metrics and print out
        test_loss /= len(data_loader)
        test_acc /= len(data_loader)

        print(f"Test loss: {test_loss:.5f} | Test acc: {test_acc:.2f}")

In [16]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
train_time_start_on_gpu = timer()

# set epochs
epochs = 10

# Create a optimization and evaluation loop using train_step and test_step
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epochs}\n-----------")
    train_step(model=model,
               data_loader=train_loader,
               loss_fn=loss_fn,
               optimizer=optimizer,
               accuracy_fn=accuracy_fn,
               device=device)
    test_step(model=model,
              data_loader=val_loader,
              loss_fn=loss_fn,
              accuracy_fn=accuracy_fn,
              device=device)

train_time_end_on_gpu = timer()
total_train_time_model = print_train_time(train_time_start_on_gpu,
                                            train_time_end_on_gpu,
                                            device = device)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 10
-----------
Train loss: 2.12199 | Train acc: 37.48%


 10%|█         | 1/10 [00:04<00:42,  4.68s/it]

Test loss: 1.02104 | Test acc: 68.45
Epoch: 10
-----------
Train loss: 0.77324 | Train acc: 74.65%


 20%|██        | 2/10 [00:09<00:36,  4.53s/it]

...

Test loss: 0.05450 | Test acc: 98.46
Epoch: 10
-----------
Train loss: 0.09677 | Train acc: 96.79%


 90%|█████████ | 9/10 [00:40<00:04,  4.47s/it]

Test loss: 0.04461 | Test acc: 98.77
Epoch: 10
-----------
Train loss: 0.09733 | Train acc: 96.89%


100%|██████████| 10/10 [00:44<00:00,  4.48s/it]

Test loss: 0.04167 | Test acc: 98.83
Tiempo de entrenamiento: 44.756 segundos en cuda


In [17]:
import os
os.makedirs("/home/uno21/models", exist_ok=True)
torch.save(model.state_dict(), "/home/uno21/models/baseline_cnn.pth")
print("Modelo guardado.")

Modelo guardado.


In [18]:
from pathlib import Path

# 1. Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path
MODEL_NAME = "baseline_cnn.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. SAve the medl state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(), f = MODEL_SAVE_PATH)

Saving model to: models/baseline_cnn.pth


In [19]:
import os
print(os.getcwd())

/tmp/pycharm_project_27c6b3c8/training
